In [2]:
import os
import sys
import torch
import yaml
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# 1. Chuyển Thư mục làm việc (CWD) về gốc dự án (Fix lỗi np.load vắng file .npy)
# Lưu ý: Chỉ chạy Cell này 1 lần duy nhất, nếu ấn chạy 2 lần nó sẽ lùi thêm 1 cấp nữa!
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
elif 'audio_event_detection' not in os.getcwd():
    os.chdir(r'H:\audio_event_detection')
print("Thư mục hiện tại:", os.getcwd())

# 2. Xóa mù đường cho lệnh 'from ... import ...'
sys.path.insert(0, os.getcwd())

from models.ast_model import AudioSpectrogramTransformer
from scripts.train import Trainer
from utils.dataset import AudioEventDataset

# 3. Khóa Random State
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


Thư mục hiện tại: h:\audio_event_detection


In [3]:
# Giờ chỉ việc dùng đường dẫn tương đối chuẩn xác, không cần ghép PROJECT_ROOT nữa
config_path = 'configs/config.yaml'
csv_path = 'data/processed/spectrograms/processed_metadata.csv'

# Đọc cấu hình từ YAML
with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

batch_size = config_dict['training']['batch_size']
num_workers = config_dict.get('hardware', {}).get('num_workers', 4)
pin_memory = config_dict.get('hardware', {}).get('pin_memory', True)

# Xác định GPU hay CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị phần cứng: {device}")


Sử dụng thiết bị phần cứng: cpu


In [4]:
print(f"Đang tải siêu dữ liệu (Metadata) từ: {csv_path}")
metadata_df = pd.read_csv(csv_path)

# Xóa các samples bị lỗi nhãn
metadata_df = metadata_df.dropna(subset=['label'])
metadata_df['label'] = metadata_df['label'].astype(int)

print(f"\nPhân phối Label trong bộ Dataset:")
print(metadata_df['target_class'].value_counts())

# Tách dữ liệu: 80% Train, 10% Val, 10% Test
# Stratify sẽ cân bằng đều các class vào cả 3 tập
train_df, temp_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nChia dữ liệu thành công!")
print(f"Số lượng Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")


Đang tải siêu dữ liệu (Metadata) từ: data/processed/spectrograms/processed_metadata.csv

Phân phối Label trong bộ Dataset:
target_class
normal            8309
dog_bark          1000
siren              929
gunshot            374
explosion           40
fire_crackling      40
scream              40
Name: count, dtype: int64

Chia dữ liệu thành công!
Số lượng Train=8585 | Val=1073 | Test=1074


In [5]:
train_dataset = AudioEventDataset(train_df, config_path, mode='train')
val_dataset = AudioEventDataset(val_df, config_path, mode='val')
test_dataset = AudioEventDataset(test_df, config_path, mode='test')

# Dùng DataLoader để nạp Data theo từng Batch vào GPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=num_workers, pin_memory=pin_memory, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                        num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)
                         
print("Tạo DataLoaders Hoàn tất!")


Tạo DataLoaders Hoàn tất!


In [6]:
print("Đang khởi tạo cấu trúc Audio Spectrogram Transformer...")
model = AudioSpectrogramTransformer(config_path)

# Nếu bạn muốn kiểm tra số lượng Parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Số lượng tham số mô hình: {total_params:,}")


Đang khởi tạo cấu trúc Audio Spectrogram Transformer...
Số lượng tham số mô hình: 85,414,664


In [ ]:
# Tạo đối tượng Huấn luyện viên (Trainer)
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config_path=config_path,
    device=str(device)
)

print("\n🚀 Bắt đầu quá trình Huấn Luyện...")
trainer.train()


In [ ]:
print("\n" + "="*60)
print("ĐÁNH GIÁ KẾT QUẢ ĐẦU RA MÔ HÌNH TRÊN TẬP TEST")
print("="*60)

# Tải Checkpoint tốt nhất được lưu tạm lúc Training
best_model_path = os.path.join(PROJECT_ROOT, config_dict['paths']['checkpoint_dir'], 'best_model.pth')

if os.path.exists(best_model_path):
    print(f"Tìm thấy Checkpoint! Đang tải Weights từ: {best_model_path}")
    checkpoint = torch.load(best_model_path, map_location=device)
    trainer.model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("Cảnh báo: Không tìm thấy best_model.pth. Sẽ test bằng trạng thái nháp hiện tại.")

# Ghi đè tập Validation bằng tập Test để tái sử dụng module Calculate loss của Trainer
trainer.val_loader = test_loader
test_metrics = trainer.validate()

print("\n[KẾT QUẢ TẬP KIỂM THỬ THỰC TẾ]")
print(f"👉 Mức độ lỗi (Loss): {test_metrics['loss']:.4f}")
if 'accuracy' in test_metrics:
    print(f"👉 Độ Chính xác đo được (Accuracy): {test_metrics['accuracy']*100:.2f}%")
if 'f1_score' in test_metrics:
    print(f"👉 Điểm số F1 (F1 Score): {test_metrics['f1_score']:.4f}")
